# pre processamento em lote

In [10]:
import cv2
import time
from pathlib import Path
import os
from tqdm import tqdm 
class DocumentProcessor:
    def __init__(self, blur_kernel=(5, 5), block_size=21, constant=10):
        self.blur_kernel = blur_kernel
        self.block_size = block_size
        self.constant = constant
        self.valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    # 1. Mostrar imagem (Útil para debug visual)
    def _mostrar_img(self, img):
        try:
            cv2.imshow("amostra", img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        except Exception as e:
            print(f"[Erro ao exibir imagem]: {e}")
    
    # 2. Leitura da imagem
    def _read_image(self, path):
        try:
            img = cv2.imread(path)
            if img is not None:
                return img
            else:
                print(f"[Aviso] Imagem não carregada ou corrompida: {path}")
                return None
        except Exception as e:
            print(f"[Erro de I/O na leitura]: {e}")
            return None

    # 3. Conversão para Escala de Cinza
    def _to_grayscale(self, img):
        try:
            return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        except Exception as e:
            print(f"[Erro na conversão de cor]: {e}")
            return None

    # 4. Tratamento de Ruído (Blur)
    def _apply_blur(self, img):
        try:
            return cv2.GaussianBlur(img, self.blur_kernel, 0)
        except Exception as e:
            print(f"[Erro na suavização gaussiana]: {e}")
            return None

    # 5. Correção de Iluminação (Threshold Adaptativo)
    def _apply_threshold(self, img):
        try:
            return cv2.adaptiveThreshold(
                img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, self.block_size, self.constant
            )
        except Exception as e:
            print(f"[Erro na limiarização]: {e}")
            return None

    # 6. Salvar o Resultado
    def _save_image(self, img, path):
        try:
            cv2.imwrite(path, img)
            return True
        except Exception as e:
            print(f"[Erro ao salvar no disco]: {e}")
            return False

    # 7. Método Principal de Processamento (Uma Imagem)
    def preprocess_img(self, input_path, output_path, mostrar=False):
        """Orquestra o passo a passo de limpeza para uma única imagem."""
        img = self._read_image(input_path)
        if img is None: return False

        gray = self._to_grayscale(img)
        if gray is None: return False

        blurred = self._apply_blur(gray)
        if blurred is None: return False

        binary = self._apply_threshold(blurred)
        if binary is None: return False

        if mostrar:
            self._mostrar_img(binary)

        return self._save_image(binary, output_path)

    # 8. Processamento em lote
    def process_batch(self, imput_dir, output_dir, limit=None):
        '''
        Processamento em lote
        '''
        try:
            t_inicial=time.perf_counter()
            #cria diretorio caso não exista
            Path(output_dir).mkdir(parents=True, exist_ok=True)
            # criar lista de caminhos das imagens de entrada
            files=[f for f in os.listdir(imput_dir) if f.lower().endswith(self.valid_extensions) ] 
            files=files[:limit]

            if not files:
                print(f"Não há imagens validas em {imput_dir}")
                return

            print(f"Iniciando preprocessamento de {len(files)} Imagens...")
            sucess=0
            for filename in tqdm(files, desc="Processando notas", unit="img"):
                in_path = os.path.join(imput_dir,filename)
                base_name = os.path.splitext(filename)[0]
                out_path = os.path.join(output_dir,f'{base_name}.png')
                if self.preprocess_img(in_path,out_path):
                    sucess+=1
            t_final=time.perf_counter()

            print(f"{sucess} Imagens processadas com sucesso em {t_final-t_inicial:.2f} segundos")
        except Exception as e:
            print(f"Erro ao processar em lote: {e}")



In [11]:
processor= DocumentProcessor()
processor.process_batch('data/raw','data/processed')

Iniciando preprocessamento de 626 Imagens...


Processando notas: 100%|██████████| 626/626 [00:42<00:00, 14.84img/s]

626 Imagens processadas com sucesso em 42.18 segundos


# Pre processamento em lote com processamento paralelo

In [7]:
import cv2
import time
from pathlib import Path
import os
from tqdm import tqdm 
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

class DocumentProcessor_concurrent:
    def __init__(self, blur_kernel=(5, 5), block_size=21, constant=10):
        self.blur_kernel = blur_kernel
        self.block_size = block_size
        self.constant = constant
        self.valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    # 1. Mostrar imagem (Útil para debug visual)
    def _mostrar_img(self, img):
        try:
            cv2.imshow("amostra", img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        except Exception as e:
            print(f"[Erro ao exibir imagem]: {e}")
    
    # 2. Leitura da imagem
    def _read_image(self, path):
        try:
            img = cv2.imread(path)
            if img is not None:
                return img
            else:
                print(f"[Aviso] Imagem não carregada ou corrompida: {path}")
                return None
        except Exception as e:
            print(f"[Erro de I/O na leitura]: {e}")
            return None

    # 3. Conversão para Escala de Cinza
    def _to_grayscale(self, img):
        try:
            return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        except Exception as e:
            print(f"[Erro na conversão de cor]: {e}")
            return None

    # 4. Tratamento de Ruído (Blur)
    def _apply_blur(self, img):
        try:
            return cv2.GaussianBlur(img, self.blur_kernel, 0)
        except Exception as e:
            print(f"[Erro na suavização gaussiana]: {e}")
            return None

    # 5. Correção de Iluminação (Threshold Adaptativo)
    def _apply_threshold(self, img):
        try:
            return cv2.adaptiveThreshold(
                img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, self.block_size, self.constant
            )
        except Exception as e:
            print(f"[Erro na limiarização]: {e}")
            return None

    # 6. Salvar o Resultado
    def _save_image(self, img, path):
        try:
            cv2.imwrite(path, img)
            return True
        except Exception as e:
            print(f"[Erro ao salvar no disco]: {e}")
            return False

    # 7. Método Principal de Processamento (Uma Imagem)
    def preprocess_img(self, input_path, output_path, mostrar=False):
        """Orquestra o passo a passo de limpeza para uma única imagem."""
        img = self._read_image(input_path)
        if img is None: return False

        gray = self._to_grayscale(img)
        if gray is None: return False

        blurred = self._apply_blur(gray)
        if blurred is None: return False

        binary = self._apply_threshold(blurred)
        if binary is None: return False

        if mostrar:
            self._mostrar_img(binary)

        return self._save_image(binary, output_path)

    # metodo auxiliar para empacotamento
    def _process_wrapper(self,filename,imput_dir, output_dir ):
        try:
            in_path = os.path.join(imput_dir,filename)
            base_name = os.path.splitext(filename)[0]
            out_path = os.path.join(output_dir,f'{base_name}.png')
            self.preprocess_img(in_path,out_path)

        except Exception as e:
            print(f'Erro ao processar {filename}: {e}')
            return False
    # 8. Processamento em lote
    def process_batch_concurrent(self, imput_dir, output_dir, limit=None):
        '''
        Processamento em lote com concorrencia de processos
        '''
        try:
            t_inicial=time.perf_counter()
            #cria diretorio caso não exista
            Path(output_dir).mkdir(parents=True, exist_ok=True)
            # criar lista de caminhos das imagens de entrada
            files=[f for f in os.listdir(imput_dir) if f.lower().endswith(self.valid_extensions) ] 
            files=files[:limit]

            if not files:
                print(f"Não há imagens validas em {imput_dir}")
                return
            
            max_workers = max(1, (os.cpu_count() or 2 )- 4)

            print(f"Iniciando preprocessamento de {len(files)} Imagens... com {max_workers} nucleos")
            sucess=0

            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                # Fixar caminhos na função parcial
                func = partial(self._process_wrapper, imput_dir=imput_dir , output_dir=output_dir)
                futures = [executor.submit(func, filename) for filename in files]    

                for future in tqdm(as_completed(futures), total=len(files), desc="Processando ...", unit="img"):
                    if future.result:
                        sucess += 1
            t_final=time.perf_counter()                        
            print(f"{sucess} Imagens processadas com sucesso em {t_final-t_inicial:.2f}")
        except Exception as e:
            print(f"Erro ao processar em lote: {e}")



In [8]:
processor_concurrent = DocumentProcessor_concurrent()
processor_concurrent.process_batch_concurrent('data/raw','data/processed_concurrent')

Iniciando preprocessamento de 626 Imagens... com 12 nucleos


Processando ...: 100%|██████████| 626/626 [00:12<00:00, 50.11img/s]

626 Imagens processadas com sucesso em 12.57
